# 🔁 PHIÊN 2 — THỰC NGHIỆM MỞ RỘNG SEQ2SEQ (BAHDANAU, WORD & SPM • KAGGLE & COLAB)
**Đề tài 8: Dịch máy Anh – Việt (IWSLT'15 En–Vi) | Môn học: Học sâu**  
**Phụ trách code: Hoàng (Phần 2) • Chạy train & W&B: Sơn (Phần 3 – Nhóm trưởng)**  
---
> **Notebook tự động nhận diện môi trường (Kaggle / Google Colab):**
> - **Trên Kaggle:** Chọn **Accelerator: GPU P100** (hoặc T4 x2), bật **Internet: ON**, vào **Add-ons → Secrets** tạo & bật secret **`BTL_DL_COLAB`** (W&B API key), sau đó bấm **Save Version → Save & Run All (Commit)**.
> - **Trên Colab:** Chọn **T4 GPU** và bật Colab Secret **`BTL_DL_COLAB`**.
>
> **Đã hoàn thành trước đó (Mốc Seq2seq Baseline):** `seq2seq_bpe_luong` — Beam(k=4) **`12.44 BLEU`** *(13.48 với `repetition_penalty=1.2`)*.
>
> **3 run Seq2seq còn lại theo đúng kế hoạch gốc (`epochs=15`, `batch_size=128`, tắt hoàn toàn spam log `tqdm`, tổng ~4.5 giờ trên P100):**
> 1. `seq2seq_bpe_bahdanau` *(So sánh Luong vs Bahdanau Attention trên BPE)*
> 2. `seq2seq_word_luong` *(So sánh Tokenizer Word-level trên Seq2seq)*
> 3. `seq2seq_spm_luong` *(So sánh Tokenizer SentencePiece trên Seq2seq)*

### Bước 1: Kiểm tra GPU & Tự nhận diện môi trường (Kaggle / Colab)

In [ ]:
!nvidia-smi

import os

IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB = (not IS_KAGGLE) and os.path.exists("/content")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/Machine_Translation_Project"
    STORAGE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"
    ENV_NAME = "Google Colab"
elif IS_KAGGLE:
    PROJECT_DIR = "/kaggle/working/Machine_Translation_Project"
    STORAGE_ROOT = "/kaggle/working/BTL_DL_Translation"
    ENV_NAME = "Kaggle"
else:
    PROJECT_DIR = os.getcwd()
    STORAGE_ROOT = os.path.join(PROJECT_DIR, "local_output")
    ENV_NAME = "Local"

os.makedirs(STORAGE_ROOT, exist_ok=True)
print(f"✓ Môi trường phát hiện: {ENV_NAME}")
print(f"  • PROJECT_DIR  : {PROJECT_DIR}")
print(f"  • STORAGE_ROOT : {STORAGE_ROOT}")

### Bước 2: Clone / Cập nhật mã nguồn nhánh `main` & Cài đặt thư viện

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/hoang180/Machine_Translation_Project.git"
BRANCH = "main"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(
    ["pip", "install", "-q", "sacrebleu", "sentencepiece", "tokenizers", "pyyaml", "wandb", "tabulate"],
    check=True,
)
print(f"✓ Đã sẵn sàng mã nguồn ({BRANCH}) và thư viện tại: {os.getcwd()}")

### Bước 3: Chuẩn bị dữ liệu `data/processed/` & 3 bộ Tokenizer

In [ ]:
import os
import shutil
import subprocess

os.chdir(PROJECT_DIR)
os.makedirs("data/processed", exist_ok=True)
for sub in ("word", "bpe", "spm"):
    os.makedirs(os.path.join("custom_tokenizers", sub), exist_ok=True)

drive_processed = os.path.join(STORAGE_ROOT, "data", "processed")
drive_tokenizers = os.path.join(STORAGE_ROOT, "custom_tokenizers")

if IS_COLAB and os.path.exists(os.path.join(drive_processed, "train.en")):
    print("[INFO] Colab: Copy data/processed & custom_tokenizers từ Google Drive...")
    shutil.copytree(drive_processed, "data/processed", dirs_exist_ok=True)
    shutil.copytree(drive_tokenizers, "custom_tokenizers", dirs_exist_ok=True)
else:
    if not os.path.exists("data/processed/train.en"):
        print("[INFO] Tải và tiền xử lý IWSLT'15 En-Vi...")
        subprocess.run(["python3", "data/download_data.py"], check=True)
        subprocess.run(["python3", "data/preprocess.py"], check=True)
    if not os.path.exists("custom_tokenizers/bpe/bpe.en.json"):
        subprocess.run(["python3", "custom_tokenizers/train_bpe.py"], check=True)
    if not os.path.exists("custom_tokenizers/spm/spm.en.model"):
        subprocess.run(["python3", "custom_tokenizers/train_spm.py"], check=True)

!ls -lh data/processed/ custom_tokenizers/word/ custom_tokenizers/bpe/ custom_tokenizers/spm/

### Bước 4: Auto-login Weights & Biases (`btl-dl-translation-iwslt15`)

In [ ]:
import os
import wandb

api_key = None
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("BTL_DL_COLAB")
    except Exception as e:
        print(f"[WARN] Không đọc được Kaggle Secret 'BTL_DL_COLAB': {e}")
elif IS_COLAB:
    try:
        from google.colab import userdata
        api_key = userdata.get("BTL_DL_COLAB")
    except Exception as e:
        print(f"[WARN] Không đọc được Colab Secret 'BTL_DL_COLAB': {e}")
else:
    api_key = os.environ.get("WANDB_API_KEY")

if api_key:
    os.environ["WANDB_API_KEY"] = api_key
    wandb.login(key=api_key, relogin=True)
    api = wandb.Api()
    print("✓ Đang đăng nhập W&B:", api.viewer.username, "| Default entity:", api.default_entity)
else:
    print("[WARN] Chưa tìm thấy Secret 'BTL_DL_COLAB' -> các run sẽ chạy không có W&B (tránh treo khi chạy nền).")

### Bước 5: Định nghĩa hàm `run_seq2seq_experiment()` (Chống spam log + Eval chuẩn & Repetition Penalty + Lưu `STORAGE_ROOT`)

In [ ]:
import json
import math
import os
import re
import subprocess
import torch


def _run_clean(cmd: list[str]) -> str:
    """Chạy subprocess, tắt hẳn tqdm và lọc mọi dòng progress bar gây tràn output."""
    print("\n>>>", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["TQDM_DISABLE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    kept_lines = []
    for line in proc.stdout:
        if "%|" in line or "it/s]" in line or "s/it]" in line:
            continue
        if "Waiting for wandb.init" in line:
            continue
        print(line, end="", flush=True)
        kept_lines.append(line)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {ret}): {' '.join(cmd)}")
    return "".join(kept_lines)


def _parse_bleu(output: str) -> dict:
    res = {}
    m = re.search(r"Điểm BLEU:\s*([0-9.]+)", output)
    if m:
        res["bleu"] = float(m.group(1))
    m_bp = re.search(r"Brevity Penalty:\s*([0-9.]+)", output)
    if m_bp:
        res["bp"] = float(m_bp.group(1))
    m_sig = re.search(r"Signature:\s*(.+)", output)
    if m_sig:
        res["signature"] = m_sig.group(1).strip()
    return res


def run_seq2seq_experiment(
    exp_dir_name: str,
    tokenizer: str,
    attention: str,
    epochs: int = 15,
    batch_size: int = 128,
    skip_if_done: bool = True,
):
    save_dir = os.path.join(STORAGE_ROOT, "checkpoints", exp_dir_name)
    ckpt_path = os.path.join(save_dir, "best_model.pt")
    eval_file = os.path.join(save_dir, "eval_result.json")

    if skip_if_done and os.path.exists(eval_file):
        print(f"[SKIP] {exp_dir_name} đã hoàn thành: {eval_file}")
        with open(eval_file, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
        return

    os.makedirs(save_dir, exist_ok=True)

    # 1. Train Seq2seq (bỏ qua nếu best_model.pt đã có sẵn)
    if skip_if_done and os.path.exists(ckpt_path):
        print(f"[RESUME] Đã thấy {ckpt_path} -> bỏ qua train, chạy thẳng eval.py!")
    else:
        _run_clean([
            "python3", "train_seq2seq.py",
            "--config", "configs/seq2seq_config.yaml",
            "--tokenizer", tokenizer,
            "--attention", attention,
            "--batch_size", str(batch_size),
            "--epochs", str(epochs),
            "--save_dir", save_dir,
        ])

    # Đọc thông tin metadata từ checkpoint
    ckpt = torch.load(ckpt_path, map_location="cpu")
    val_loss = float(ckpt.get("val_loss", 0.0))
    best_epoch = int(ckpt.get("epoch", 0))
    total_params = sum(v.numel() for v in ckpt["model_state_dict"].values())
    del ckpt

    # 2. Eval Beam Search (k=4, alpha=0.6) — cả chuẩn (rep=1.0) và chống lặp từ (rep=1.2)
    print(f"\n[INFO] Đang chạy eval.py trên tst2013 cho {exp_dir_name}...", flush=True)
    out_beam4 = _run_clean([
        "python3", "eval.py", "--checkpoint", ckpt_path,
        "--beam_size", "4", "--alpha", "0.6", "--repetition_penalty", "1.0"
    ])
    out_beam4_rp = _run_clean([
        "python3", "eval.py", "--checkpoint", ckpt_path,
        "--beam_size", "4", "--alpha", "0.6", "--repetition_penalty", "1.2"
    ])

    eval_result = {
        "exp_dir": exp_dir_name,
        "run_name": f"seq2seq-{attention}-{tokenizer}",
        "tokenizer": tokenizer,
        "attention": attention,
        "trainable_params": total_params,
        "best_epoch": best_epoch,
        "val_loss": round(val_loss, 4),
        "val_ppl": round(math.exp(min(val_loss, 100.0)), 2),
        "beam4": _parse_bleu(out_beam4),
        "beam4_rep12": _parse_bleu(out_beam4_rp),
    }
    with open(eval_file, "w", encoding="utf-8") as f:
        json.dump(eval_result, f, ensure_ascii=False, indent=2)

    print(f"\n✓ HOÀN TẤT & ĐÃ LƯU TẠI: {save_dir}")
    print(
        f"  -> BLEU Beam(k=4): {eval_result['beam4'].get('bleu')} | "
        f"Beam(k=4, rep=1.2): {eval_result['beam4_rep12'].get('bleu')}"
    )

print(f"✓ Đã sẵn sàng hàm run_seq2seq_experiment() trên {ENV_NAME}!")

---
### Run 1/3 — Seq2seq Bahdanau Attention (BPE): So sánh Luong vs Bahdanau

In [ ]:
run_seq2seq_experiment(
    exp_dir_name="seq2seq_bpe_bahdanau",
    tokenizer="bpe",
    attention="bahdanau",
    epochs=15,
    batch_size=128,
)

### Run 2/3 — Seq2seq Luong Attention (Word Tokenizer)

In [ ]:
run_seq2seq_experiment(
    exp_dir_name="seq2seq_word_luong",
    tokenizer="word",
    attention="luong",
    epochs=15,
    batch_size=128,
)

### Run 3/3 — Seq2seq Luong Attention (SentencePiece Tokenizer)

In [ ]:
run_seq2seq_experiment(
    exp_dir_name="seq2seq_spm_luong",
    tokenizer="spm",
    attention="luong",
    epochs=15,
    batch_size=128,
)

---
### Tổng hợp đủ 4/4 runs Seq2seq ra CSV & JSON (chèn vào báo cáo ACL)

In [ ]:
import json
import os
import pandas as pd

CKPT_ROOT = os.path.join(STORAGE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(STORAGE_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEQ2SEQ_EXPERIMENTS = [
    ("seq2seq_bpe_luong",    "Seq2seq Luong (BPE 16k — Baseline)"),
    ("seq2seq_bpe_bahdanau", "Seq2seq Bahdanau (BPE 16k)"),
    ("seq2seq_word_luong",   "Seq2seq Luong (Word 25k/11.8k)"),
    ("seq2seq_spm_luong",    "Seq2seq Luong (SentencePiece 16k/13k)"),
]

# Thông tin mốc Baseline đã hoàn thành trên Colab (notebook 01)
BASE_SEQ2SEQ_ROW = {
    "Config": "Seq2seq Luong (BPE 16k — Baseline)",
    "Run": "seq2seq-luong-bpe",
    "Attention": "luong",
    "Tokenizer": "bpe",
    "Params": 31892608,
    "Best_Epoch": 15,
    "Val_Loss": 4.7483,
    "Val_PPL": 115.39,
    "Test_BLEU_Beam4": 12.44,
    "Test_BLEU_Beam4_Rep12": 13.48,
}

rows = []
for folder, label in SEQ2SEQ_EXPERIMENTS:
    e_path = os.path.join(CKPT_ROOT, folder, "eval_result.json")
    if not os.path.exists(e_path):
        if folder == "seq2seq_bpe_luong":
            rows.append(BASE_SEQ2SEQ_ROW)
        continue
    with open(e_path, "r", encoding="utf-8") as f:
        ev = json.load(f)
    rows.append({
        "Config": label,
        "Run": ev.get("run_name", folder),
        "Attention": ev.get("attention"),
        "Tokenizer": ev.get("tokenizer"),
        "Params": ev.get("trainable_params"),
        "Best_Epoch": ev.get("best_epoch"),
        "Val_Loss": ev.get("val_loss"),
        "Val_PPL": ev.get("val_ppl"),
        "Test_BLEU_Beam4": ev.get("beam4", {}).get("bleu"),
        "Test_BLEU_Beam4_Rep12": ev.get("beam4_rep12", {}).get("bleu"),
    })

df = pd.DataFrame(rows)
csv_out = os.path.join(RESULTS_DIR, "seq2seq_experiments_summary.csv")
json_out = os.path.join(RESULTS_DIR, "seq2seq_experiments_summary.json")
df.to_csv(csv_out, index=False, encoding="utf-8")
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"✓ Đã xuất bảng tổng hợp ({len(rows)}/4 runs) ra:\n  - {csv_out}\n  - {json_out}\n")
print(df.to_markdown(index=False))